# Sizing a myLedger node

**Answers:** how much memory, how much disk, and how many device reads a second — for a deployment
nobody has run.

**Does not answer:** throughput and tail latency. Those are not arithmetic. Design notes §10 records
a formula for them being built, measured and *refused* — the stages' cache misses overlap and it
priced each at full latency. `ledgersim` runs the real reactor on a virtual clock instead.

## Who owns what

| | owns | why |
|---|---|---|
| the code | what one unit costs | it is `size_of`, known at build time |
| this notebook | how many units | it follows from a rate, a lifetime and a retention |

No byte count is written down here. They come from `sizing/units.json` — `ledgerfio layout --json`
with the commit it was taken at. `make sizing-units` refreshes it; `make verify` fails if it is stale.

> If a cell fails with a name that should exist, **Restart Kernel and Run All**. Cell 1 reloads
> `model.py` for exactly this reason, but a half-run notebook still has half-old globals.


In [ ]:
# Find `model.py` whichever directory the kernel started in: VS Code opens a notebook at its own
# folder by default, but `jupyter.notebookFileRoot` can be set to the workspace root, and then a
# bare '.' is the wrong place. Searching removes the setting from the picture.
import importlib, os, sys

here = os.getcwd()
while not os.path.exists(os.path.join(here, 'model.py')):
    beside = os.path.join(here, 'sizing')
    if os.path.exists(os.path.join(beside, 'model.py')):
        here = beside
        break
    parent = os.path.dirname(here)
    if parent == here:
        raise SystemExit('open this notebook from the repository, beside sizing/model.py')
    here = parent
sys.path.insert(0, here)

# Reload rather than import: a kernel keeps the first `model` it saw in `sys.modules`, so editing
# model.py and re-running this cell would otherwise get the old one back -- and the failure lands
# several cells later as a name that does not exist, which reads like the notebook being wrong.
import model
importlib.reload(model)
from model import (Load, Lifetimes, Demand, Policy, Dials, Sizing, report, gigabytes,
                   residency_curve, print_residency_curve,
                   flush_window_curve, print_flush_window_curve,
                   print_unit_costs, buckets_for, index_slots_for,
                   check_bucket_rule, load_units)

units = load_units()
check_bucket_rule(units)   # the one piece of the code's arithmetic reproduced here
print(units['source'], 'at', units['commit'][:12])
print(len(units['parts']), 'sized structures')


## 0. What one of each thing costs

These come from the build — `size_of`, published by `ledgerfio layout --json` and cached in
`units.json`. **Nothing below hard-codes one.**

`overrides` is how you ask *what if this struct were smaller*. Put a name and a byte count in it and
every number in this notebook follows, including the disk figure — records per block is derived from
the record size, not read off the dump. Anything overridden is marked here and again in the report,
because a hypothetical printed beside measurements reads as a measurement.


In [ ]:
overrides = {}
# overrides = {'pending record': 52}   # e.g. what if a hold's record lost another field?
# overrides = {'lane state': 16}       # e.g. what if a lane were half as wide?

print_unit_costs(units, overrides)


## 1. The load curve — how much traffic, at every window width

**A day is not flat, and this ledger has windows of several different widths.** So traffic is a
curve: *how much goes through the busiest window this wide*.

It replaced four numbers — a peak rate, how long the peak lasts, the busiest hour, the day — which
were four readings of one curve, free to disagree. Two shortcuts were in this file and both were
wrong:

- *busiest hour × 4* for a four-hour window assumes four equally busy hours in a row. **Over-sizes.**
- *day ÷ 24 × 2* for a two-hour window assumes a flat day. **Under-sizes.**

Each structure asks this curve **at its own window's width**, so changing a window changes what it
is sized by. That is the thing four fixed numbers could not express.


In [ ]:
load = Load([
    (1/60, 200_000 * 60),   # the busiest MINUTE: 200k/s, the peak rate
    (1,     30_000_000),    # the busiest HOUR: a tenth of the day
    (4,     90_000_000),    # the busiest FOUR hours: a third of the day
    (24,   300_000_000),    # the DAY
])

for hours in (1/60, 1, 4, 12, 24):
    tx = load.busiest(hours)
    print(f'busiest {hours:>6.2f}h  {tx:>14,.0f} tx  ({tx/(hours*3600):>9,.0f}/s average)')


## 2. The lifetime curve — how long holds live

One sentence a business can say — *half are voided within the hour* — answers three questions this
model used to guess at separately:

| read at | tells you |
|---|---|
| the flush window | what reaches the disk at all |
| residency | what a resolution costs |
| retention | what expiry has to void |

The mean hold life Little's law wants is the area above this curve, rather than a number nobody
derived.


In [ ]:
lifetimes = Lifetimes([
    (1,      0.50),   # half are settled or voided within the hour
    (4,      0.70),
    (24,     0.88),
    (24 * 7, 0.96),   # the rest never resolve -- which is what retention is for
])

for hours in (1, 4, 24, 24 * 30):
    print(f'{hours:>5}h  {lifetimes.resolved_by(hours):>5.0%} resolved')


## 3. The three scalars

**`hold_share`** — of every transaction, the share that *creates* a hold. Count a day:

| | per day | creates a hold? |
|---|---|---|
| authorisation (card tapped) | 270M | **yes** |
| capture (money actually moves) | — | no, it removes one |
| plain transfer | 30M | no |

The index counts *live holds*, so it is 270M that matters, not 300M.

**`records_per_hold`** — records one hold appends over its life. Append-only means a changed
remainder is a **new record**, not an edit:

- 100,000 authorised → 1 record
- captured in full → **0** more (a removal is not a record)
- captured 30,000 then 70,000 → **1** more (the first leaves a remainder; the second removes)

So it is `1 + partial captures that leave something behind`. 1.2 means one hold in five is split.

**`commit_latency_seconds`** — submit to commit, mostly the consensus round trip. Times the peak
rate it is *requests in the air at once*: 200,000/s × 10ms = 2,000. That is all it sizes — the slot
pool. It is an input here rather than an answer, because latency is `ledgersim`'s question.


In [ ]:
demand = Demand(
    load=load,
    lifetimes=lifetimes,
    accounts=100_000_000,        # the working set. accounts never leave, so no rate enters this
    hold_share=0.90,             # 90% of transactions create a hold
    records_per_hold=1.2,        # one hold in five is captured in two parts
    commit_latency_seconds=0.010,
)
print(demand.sanity() or 'inputs are consistent')
print(f'peak {demand.peak_rate:,.0f}/s, day {demand.daily_tx:,.0f} tx')


## 4. Policy — four bounds, each with a name

| | is a bound on | widening it costs | widening it buys |
|---|---|---|---|
| `retention_days` | **how long a hold may live** before expiry voids it | disk, and the index | a longer promise to the customer |
| `flush_window_hours` | **recovery** — what a restart replays | buffer memory, restart time | *less disk*, see §6 |
| `residency_hours` | **latency** — how far back memory answers | resident blocks | *fewer device reads*, see §5 |
| `idem_window_hours` | **duplicate detection** — how late a retry may arrive | memory, linearly | tolerance for late retries |

Retention is the first decision, because everything on disk is multiplied by it.

None of these has to be the number below. Every one is swept later in this notebook, and each sweep
asks the load curve at the new width — which is why the windows being variable is expressible at all.


In [ ]:
policy = Policy(
    retention_days=30,
    grace_days=1,                # slack before deletion, so expiry is never early
    flush_window_hours=1,
    residency_hours=24,
    idem_window_hours=1,         # the code does not enforce this yet -- the map only grows
    snapshot_every_effects=1_000_000,
)

sizing = Sizing(demand, policy, Dials(), overrides=overrides)
print(report(sizing))


## 5. Residency is an answer, not a preference

A resolution whose record is still in memory is free; one past the residency window **reads the**
**device**. The lifetime curve says when resolutions arrive, so the read rate is a subtraction on
it — resolutions landing between residency ending and expiry taking the hold.

So the question is not *how many hours feels right*. It is **how many device reads a second you are**
**willing to pay for**, and the hours follow.

Printed as a curve rather than solved: what a read is worth depends on the device and on the tail
somebody is holding, and neither is in this file.


In [ ]:
print_residency_curve(residency_curve(demand, policy, [1, 2, 4, 8, 24, 72, 168],
                                      units=units))


## 6. The flush window trades memory for disk — in both directions

A record whose hold resolves **before its block is flushed** is dropped by compaction and never
written. Half these holds resolve within the hour, so an hour-wide window already discards half the
records.

Widening it is therefore *more* buffer memory and *less* disk. But the memory curve is not monotone:
a narrower window writes more to disk, and what is written is also what residency has to keep — so a
very short window costs memory at both ends. There is an interior minimum.

What it costs that is not on this table: **recovery time**. The flush window is how much a restart
replays, and nothing here prices that.


In [ ]:
print_flush_window_curve(flush_window_curve(demand, policy, [0.25, 0.5, 1, 2, 4, 12, 24],
                                            units=units))


## 7. Retention moves everything on disk


In [ ]:
print(f"{'retention':>10}{'expiry voids':>14}{'live holds':>16}{'index GB':>10}{'disk GB':>10}")
for days in (7, 10, 30, 90, 180):
    one = Sizing(demand, Policy(retention_days=days, grace_days=1,
                                flush_window_hours=policy.flush_window_hours,
                                residency_hours=policy.residency_hours,
                                idem_window_hours=policy.idem_window_hours))
    index = dict(one.lines_by_name)['pending index']
    print(f'{days:>9}d{one.survivor_share:>14.1%}{one.live_holds:>16,}'
          f'{gigabytes(index.bytes):>10.2f}{gigabytes(one.disk_bytes):>10.1f}')


## 8. The staircase

A hash table rounds its bucket count to a power of two, so **one percent more entries can double**
**the memory**. The cuckoo index steps too, four times more coarsely. A single point says nothing
about which side of a step it is standing on.


In [ ]:
print(f"{'daily tx':>14}{'live holds':>16}{'index slots':>16}{'index GB':>11}")
for daily in (100, 200, 300, 400, 600, 800):
    scaled = Load([(h, tx * daily / 300) for h, tx in load.points])
    one = Sizing(Demand(scaled, lifetimes, demand.accounts, demand.hold_share,
                        demand.records_per_hold), policy)
    line = dict(one.lines_by_name)['pending index']
    print(f'{daily * 1_000_000:>14,}{one.live_holds:>16,}{line.count:>16,}'
          f'{gigabytes(line.bytes):>11.2f}')


### The idem map, and why the peak's *duration* matters

The idem window has no queue to absorb a peak into: its count is simply what went through the
window. This is where the shape of the day decides everything.


In [ ]:
print(f"{'idem window':>13}{'tx in it':>16}{'buckets':>16}{'idem GB':>10}")
for window in (0.25, 1, 4, 12, 24):
    one = Sizing(demand, Policy(retention_days=policy.retention_days, grace_days=1,
                                flush_window_hours=policy.flush_window_hours,
                                residency_hours=policy.residency_hours,
                                idem_window_hours=window))
    line = dict(one.lines_by_name)['idem keys']
    print(f'{window:>12}h{load.busiest(window):>16,.0f}{line.count:>16,}'
          f'{gigabytes(line.bytes):>10.2f}')


## Changing a unit cost

Set it in `overrides` at the top and re-run — every number here follows, including the disk figure.
Below is the same thing done inline, to show what the marking looks like when it fires.


In [ ]:
what_if = Sizing(demand, policy, Dials(), overrides={'pending record': 52})
print(report(what_if).splitlines()[0])
print(f'measured   {gigabytes(sizing.disk_bytes):>7.1f} GB of disk at {sizing.records_per_block} records a block')
print(f'overridden {gigabytes(what_if.disk_bytes):>7.1f} GB       at {what_if.records_per_block} records a block')


## What is not sized here

- **Throughput and the tail** — `ledgersim capacity` and `ledgersim require`, which run the real
  reactor rather than a formula.
- **`skew`** — hot-account concentration costs lane contention, which is latency, not bytes.
- **Recovery time** — the flush window sets it and nothing here prices it.
- **The idem window** — one hour is the intended window and §6's table assumes it, but the rotating
  generations that would enforce it are not built. Today the map only grows.
- **`kept log`** — no compaction, so its count is a snapshot cadence rather than a steady state.
